# 01 — Exploring and cleaning the population data (INS TEMPO)

Goal: turn 42 raw CSV exports into a single clean table, one row per locality.

This notebook is the exploration trail. The production version of the same logic lives in
`src/prep_population.py`, which is what actually produces the file used downstream.

Source column names (`Localitati`, `Varste si grupe de varsta`, `Judete`, `Valoare`) are kept
exactly as INS exports them, so every figure stays traceable to the raw file.

## 1. Read every file in `data/raw/population/` with `glob`

In [1]:
import glob
import pandas as pd


def read_csv_files(file_pattern):
    """Read every CSV matching the pattern and concatenate them into one DataFrame."""
    csv_files = sorted(glob.glob(file_pattern))
    if not csv_files:
        raise FileNotFoundError(f"no files matched the pattern {file_pattern!r}")

    dataframes = []
    for file in csv_files:
        # skipinitialspace: the TEMPO export puts a space after every comma
        dataframes.append(pd.read_csv(file, skipinitialspace=True))

    return pd.concat(dataframes, ignore_index=True)

## 2. Concatenate into a single DataFrame

In [2]:
df = read_csv_files("../data/raw/population/*.csv")

In [3]:
df.head(10)

,Varste si grupe de varsta,Sexe,Judete,Localitati,Ani,UM: Numar persoane,Valoare
0,Total,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,2108049
1,65-69 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,126370
2,70-74 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,122788
3,75-79 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,81809
4,80-84 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,41639
5,85 ani si peste,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,53995
6,Total,Total,Alba,1017 MUNICIPIUL ALBA IULIA,Anul 2026,Numar persoane,74137
7,Total,Total,Alba,1213 MUNICIPIUL AIUD,Anul 2026,Numar persoane,23600
8,Total,Total,Alba,1348 MUNICIPIUL BLAJ,Anul 2026,Numar persoane,19821
9,Total,Total,Alba,1874 MUNICIPIUL SEBES,Anul 2026,Numar persoane,31880


In [4]:
df.shape

(19086, 7)

In [5]:
df.columns.tolist()

['Varste si grupe de varsta',
 'Sexe',
 'Judete',
 'Localitati ',
 'Ani',
 'UM: Numar persoane',
 'Valoare']

## 3. Clean the column names

Note: `'Localitati '` carries a trailing space. `skipinitialspace` removes the space *after*
a comma, not the one before it, so this has to be handled separately.

In [6]:
df.columns = df.columns.str.strip()

In [7]:
df.columns.tolist()

['Varste si grupe de varsta',
 'Sexe',
 'Judete',
 'Localitati',
 'Ani',
 'UM: Numar persoane',
 'Valoare']

## 4. Split the SIRUTA code from the locality name

`2130 ALBAC` versus `1017 MUNICIPIUL ALBA IULIA` — one rule has to work for both.
A single cut at the first space does it, which is what `n=1` means. The code stays text:
it is an identifier, not a quantity.

In [8]:
df[["siruta_code", "locality"]] = df["Localitati"].str.split(" ", n=1, expand=True)

In [9]:
df.head()

,Varste si grupe de varsta,Sexe,Judete,Localitati,Ani,UM: Numar persoane,Valoare,siruta_code,locality
0,Total,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,2108049,179132,MUNICIPIUL BUCURESTI
1,65-69 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,126370,179132,MUNICIPIUL BUCURESTI
2,70-74 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,122788,179132,MUNICIPIUL BUCURESTI
3,75-79 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,81809,179132,MUNICIPIUL BUCURESTI
4,80-84 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,41639,179132,MUNICIPIUL BUCURESTI


In [10]:
df.dtypes

Varste si grupe de varsta      str
Sexe                           str
Judete                         str
Localitati                     str
Ani                            str
UM: Numar persoane             str
Valoare                      int64
siruta_code                    str
locality                       str
dtype: object

## 5. Pivot from long to wide

One row per locality, one column per age group.

`pivot` rather than `pivot_table`: the latter silently averages duplicates, while `pivot`
raises instead — which validates for free that every locality appears exactly once per
age group.

In [11]:
wide = df.pivot(
    index=["Judete", "siruta_code", "locality"],
    columns="Varste si grupe de varsta",
    values="Valoare",
)
wide.columns.name = None
wide = wide.reset_index()
wide.shape

(3181, 9)

In [12]:
wide.columns.tolist()

['Judete',
 'siruta_code',
 'locality',
 '65-69 ani',
 '70-74 ani',
 '75-79 ani',
 '80-84 ani',
 '85 ani si peste',
 'Total']

## 6. Rename to stable English column names

The age-group labels are INS *values* that became columns during the pivot. From here on the
pipeline owns them, so they get stable identifiers. The 33 health-unit category names in the
other source stay verbatim, because those remain columns of the source itself.

In [13]:
wide = wide.rename(columns={
    "Judete": "county",
    "65-69 ani": "age_65_69",
    "70-74 ani": "age_70_74",
    "75-79 ani": "age_75_79",
    "80-84 ani": "age_80_84",
    "85 ani si peste": "age_85_plus",
    "Total": "population_total",
})
wide.columns.tolist()

['county',
 'siruta_code',
 'locality',
 'age_65_69',
 'age_70_74',
 'age_75_79',
 'age_80_84',
 'age_85_plus',
 'population_total']

## 7. Derived columns: `population_65plus` and `share_65plus`

`axis=1` sums across the row, one value per locality. `population_total` is deliberately left
out of the sum — it is the denominator, not one of the parts.

`share_65plus` stays a 0–1 ratio: percent formatting belongs to the presentation layer. Storing
it on a 0–100 scale and then applying a percentage format in Power BI produces `1880%`.

In [14]:
AGE_GROUPS_65_PLUS = ["age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85_plus"]

wide["population_65plus"] = wide[AGE_GROUPS_65_PLUS].sum(axis=1)
wide["share_65plus"] = wide["population_65plus"] / wide["population_total"]
wide.head()

,county,siruta_code,locality,age_65_69,age_70_74,age_75_79,age_80_84,age_85_plus,population_total,population_65plus,share_65plus
0,Alba,1017,MUNICIPIUL ALBA IULIA,5082,4639,2698,1227,994,74137,14640,0.197472
1,Alba,1071,CIUGUD,179,208,136,74,54,3448,651,0.188805
2,Alba,1151,ORAS ABRUD,331,296,179,110,87,4829,1003,0.207703
3,Alba,1213,MUNICIPIUL AIUD,1736,1686,1022,510,477,23600,5431,0.230127
4,Alba,1348,MUNICIPIUL BLAJ,1239,1221,824,454,359,19821,4097,0.206700


## 8. Checks

Expected values are declared once, so an error message cannot contradict its own condition.

The last two are the valuable ones: they were computed independently with `awk`, straight from
the raw CSVs, without pandas. Two implementations sharing no code that agree on the same number
is a far stronger signal than any single pipeline checking itself.

In [15]:
EXPECTED_LOCALITIES = 3181
EXPECTED_COLUMNS = 11
EXPECTED_TOTAL_65_PLUS = 4_076_589
EXPECTED_TOTAL_POPULATION = 21_646_220

# 1. no locality lost, none invented
assert len(wide) == EXPECTED_LOCALITIES, \
    f"expected {EXPECTED_LOCALITIES} localities, got {len(wide)}"

# 2. the table has the shape we expect
assert len(wide.columns) == EXPECTED_COLUMNS, \
    f"expected {EXPECTED_COLUMNS} columns, got {len(wide.columns)}: {wide.columns.tolist()}"

# 3. the join key really is a key — if it breaks, the join with the health-unit
#    source multiplies rows silently instead of failing
duplicated = wide["siruta_code"].duplicated()
assert not duplicated.any(), (
    f"{duplicated.sum()} duplicate SIRUTA codes: "
    f"{wide.loc[wide['siruta_code'].duplicated(keep=False), 'siruta_code'].unique()[:10].tolist()}"
)

# 4. no missing values — sum(axis=1) treats NaN as zero and would have hidden them
missing = wide.isna().sum()
assert missing.sum() == 0, f"missing values by column:\n{missing[missing > 0].to_string()}"

# 5. logical invariant: a subset cannot exceed the whole
over = ~wide["population_65plus"].between(0, wide["population_total"])
assert not over.any(), (
    f"{over.sum()} localities where population_65plus falls outside [0, population_total]:\n"
    f"{wide.loc[over, ['locality', 'population_65plus', 'population_total']].head(10).to_string()}"
)

# 6. a share is a ratio, so it belongs to [0, 1]; also catches inf from a zero denominator
outside = ~wide["share_65plus"].between(0, 1)
assert not outside.any(), (
    f"{outside.sum()} localities where share_65plus falls outside [0, 1]:\n"
    f"{wide.loc[outside, ['locality', 'population_total', 'share_65plus']].head(10).to_string()}"
)

# 7-8. cross-check against the independent awk implementation on the raw CSVs
assert wide["population_65plus"].sum() == EXPECTED_TOTAL_65_PLUS, \
    f"total 65+: expected {EXPECTED_TOTAL_65_PLUS:,}, got {wide['population_65plus'].sum():,}"

assert wide["population_total"].sum() == EXPECTED_TOTAL_POPULATION, \
    f"total population: expected {EXPECTED_TOTAL_POPULATION:,}, got {wide['population_total'].sum():,}"

print(f"✓ {len(wide):,} localities · {EXPECTED_COLUMNS} columns · 8 checks passed")

✓ 3,181 localities · 11 columns · 8 checks passed


## 9. Exclude Bucharest

The reason is technical, not thematic: Bucharest arrives aggregated, 2.1 million people in a
single row with no breakdown by sector, which makes it incomparable with the remaining 3,180
territorial units (median population 3,003).

The checks above run on the complete set, *before* this exclusion — a fully validated set is a
stronger baseline, and it keeps the national totals available for comparison.

In [16]:
BUCHAREST_SIRUTA = "179132"

working = wide[wide["siruta_code"] != BUCHAREST_SIRUTA]

# Did the filter remove exactly one row?
# A filter that fails to filter raises nothing — it would leave Bucharest (2.1M people)
# in every aggregate and corrupt the result silently.
removed = len(wide) - len(working)
assert removed == 1, (
    f"the filter should have removed exactly 1 row (Bucharest), it removed {removed}. "
    f"Check the type and format of siruta_code: {wide['siruta_code'].dtype}"
)

print(f"✓ {len(wide):,} -> {len(working):,} localities "
      f"({working['population_total'].sum():,} people, "
      f"{working['population_65plus'].sum() / working['population_total'].sum():.1%} aged 65+)")

✓ 3,181 -> 3,180 localities (19,538,171 people, 18.7% aged 65+)


## 10. Export

The result is saved to `data/processed/` so the next steps do not depend on running this
notebook. `src/prep_population.py` produces the same file from the command line.

In [17]:
from pathlib import Path

OUTPUT = Path("../data/processed/population_by_locality.csv")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

working.to_csv(OUTPUT, index=False)

# Round-trip check: does what we wrote read back identically?
# CSV does not preserve dtypes — siruta_code would come back as int64 and break the join
# with the health-unit source, silently. Hence the explicit dtype on read.
check = pd.read_csv(OUTPUT, dtype={"siruta_code": "str"})
assert len(check) == len(working), f"wrote {len(working)} rows, read back {len(check)}"
assert check["population_total"].sum() == working["population_total"].sum(), \
    "totals do not match after export"

print(f"✓ {OUTPUT} — {len(check):,} rows, {OUTPUT.stat().st_size / 1024:.0f} KB")

✓ ../data/processed/population_by_locality.csv — 3,180 rows, 219 KB
